In [2]:
import pandas as pd

orders = pd.read_csv(
    "../data/processed/cleaned_online_retail.csv"
)

optimized = pd.read_csv(
    "../data/processed/optimized_slot_recommendations.csv"
)

warehouse_slots = pd.read_csv(
    "../data/processed/warehouse_slots.csv"
)

product_base = pd.read_csv(
    "../data/processed/final_slot_recommendations.csv"
)

C:\Users\HP\AppData\Local\Temp\ipykernel_22616\3769873957.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  orders = pd.read_csv(


In [3]:
product_base = product_base.reset_index(drop=True)

product_base["current_slot"] = [
    warehouse_slots.iloc[i % len(warehouse_slots)]["slot"]
    for i in range(len(product_base))
]

In [4]:
slot_coordinates = warehouse_slots.set_index("slot")[
    ["x", "y"]
].to_dict("index")

product_base["current_x"] = product_base["current_slot"].map(
    lambda x: slot_coordinates[x]["x"]
)

product_base["current_y"] = product_base["current_slot"].map(
    lambda x: slot_coordinates[x]["y"]
)

In [5]:
current_locations = product_base[
    ["StockCode", "current_x", "current_y"]
].copy()

In [6]:
optimized_locations = optimized[
    ["StockCode", "optimized_x", "optimized_y"]
].copy()

In [7]:
orders = orders[
    ["InvoiceNo", "StockCode"]
].drop_duplicates()

In [8]:
orders_current = orders.merge(
    current_locations,
    on="StockCode",
    how="left"
)

In [9]:
packing_x = 0
packing_y = 0

In [10]:
orders_current["current_distance"] = (
    abs(orders_current["current_x"] - packing_x)
    +
    abs(orders_current["current_y"] - packing_y)
)

In [11]:
current_order_distance = (
    orders_current
    .groupby("InvoiceNo")["current_distance"]
    .sum()
    .reset_index()
)

current_order_distance = current_order_distance.rename(
    columns={
        "current_distance": "before_distance"
    }
)

In [12]:
orders_optimized = orders.merge(
    optimized_locations,
    on="StockCode",
    how="left"
)

In [13]:
orders_optimized["optimized_distance"] = (
    abs(orders_optimized["optimized_x"] - packing_x)
    +
    abs(orders_optimized["optimized_y"] - packing_y)
)

In [14]:
optimized_order_distance = (
    orders_optimized
    .groupby("InvoiceNo")["optimized_distance"]
    .sum()
    .reset_index()
)

optimized_order_distance = optimized_order_distance.rename(
    columns={
        "optimized_distance": "after_distance"
    }
)

In [15]:
evaluation = current_order_distance.merge(
    optimized_order_distance,
    on="InvoiceNo",
    how="inner"
)

In [16]:
evaluation["distance_saved"] = (
    evaluation["before_distance"]
    - evaluation["after_distance"]
)

In [17]:
evaluation["improvement_percent"] = (
    evaluation["distance_saved"]
    / evaluation["before_distance"]
) * 100

In [18]:
print(evaluation.head(20))

   InvoiceNo  before_distance  after_distance  distance_saved  \
0     536365               23             2.0            21.0   
1     536366                9             0.0             9.0   
2     536367               48             0.0            48.0   
3     536368               17             7.0            10.0   
4     536369                2             0.0             2.0   
5     536370               71             0.0            71.0   
6     536371                5             1.0             4.0   
7     536372                9             0.0             9.0   
8     536373               62             9.0            53.0   
9     536374                2             0.0             2.0   
10    536375               62             9.0            53.0   
11    536376                8             0.0             8.0   
12    536377                9             0.0             9.0   
13    536378               70             6.0            64.0   
14    536380             

In [19]:
before_avg = evaluation["before_distance"].mean()
after_avg = evaluation["after_distance"].mean()

distance_saved_avg = before_avg - after_avg

improvement = (
    distance_saved_avg / before_avg
) * 100

print(f"Average Before Distance: {before_avg:.2f}")
print(f"Average After Distance: {after_avg:.2f}")
print(f"Average Distance Saved: {distance_saved_avg:.2f}")
print(f"Improvement: {improvement:.2f}%")

Average Before Distance: 102.94
Average After Distance: 3.94
Average Distance Saved: 99.00
Improvement: 96.17%


In [20]:
relocated_products = (
    optimized[
        optimized["optimized_slot"].notna()
    ]["StockCode"]
    .nunique()
)

print(
    f"Products Relocated: {relocated_products}"
)

Products Relocated: 25


In [21]:
evaluation.to_csv(
    "../data/processed/before_after_evaluation.csv",
    index=False
)

In [22]:
summary = pd.DataFrame({
    "Metric": [
        "Average Before Distance",
        "Average After Distance",
        "Average Distance Saved",
        "Improvement %",
        "Products Relocated"
    ],
    "Value": [
        before_avg,
        after_avg,
        distance_saved_avg,
        improvement,
        relocated_products
    ]
})

print(summary)

                    Metric       Value
0  Average Before Distance  102.943593
1   Average After Distance    3.939385
2   Average Distance Saved   99.004208
3            Improvement %   96.173259
4       Products Relocated   25.000000


In [23]:
summary.to_csv(
    "../data/processed/optimization_summary.csv",
    index=False
)